In [ ]:
import shutil
from pathlib import Path

import numpy as np
from ultralytics import YOLO

MODEL_PATH = "../runs/runs/baseline_yolo11s_e100/runs/kaggle_yolo11s_optimized/weights/best.pt"
WEB_MODEL_DIR = Path("../web/model")
WEB_MODEL_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO(MODEL_PATH)
onnx_path = model.export(format="onnx", imgsz=640, nms=True, simplify=True)
print(f"exported to {onnx_path}")

Ultralytics 8.4.104 🚀 Python-3.11.9 torch-2.13.0 CPU (Apple M1 Pro)


YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs



PyTorch: starting from '../runs/runs/baseline_yolo11s_e100/runs/kaggle_yolo11s_optimized/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (18.3 MB)


requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...



   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [onnxslim]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnxslim]


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/dmytroborbotko/ai-projects/real-time-uav-detection/.venv/bin/python -m pip install --upgrade pip



...


ONNX: starting export with onnx 1.22.0 opset 20...


ONNX: slimming with onnxslim 0.1.95...


ONNX: export success ✅ 4.4s, saved as '../runs/runs/baseline_yolo11s_e100/runs/kaggle_yolo11s_optimized/weights/best.onnx' (36.2 MB)



Export complete (5.6s)
Results saved to /Users/dmytroborbotko/ai-projects/real-time-uav-detection/runs/runs/baseline_yolo11s_e100/runs/kaggle_yolo11s_optimized/weights/best.onnx
Predict:         yolo predict task=detect model=../runs/runs/baseline_yolo11s_e100/runs/kaggle_yolo11s_optimized/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=../runs/runs/baseline_yolo11s_e100/runs/kaggle_yolo11s_optimized/weights/best.onnx imgsz=640 data=/kaggle/working/data.yaml  
Visualize:       https://netron.app


exported to ../runs/runs/baseline_yolo11s_e100/runs/kaggle_yolo11s_optimized/weights/best.onnx


In [2]:
dest = WEB_MODEL_DIR / "best.onnx"
shutil.copyfile(onnx_path, dest)
print(f"copied to {dest} ({dest.stat().st_size / 1e6:.1f} MB)")

copied to ../web/model/best.onnx (38.0 MB)


## Sanity check: PyTorch vs ONNX output on the same frame

Mirrors Day 5's original DoD ("вивід збігається") — catches export bugs (wrong opset, botched
NMS fusion, preprocessing mismatch) before any of this is ported to JS. Loads the exported ONNX
file back through `ultralytics.YOLO` (it natively supports an ONNX backend), so both runs go
through identical, already-trusted pre/post-processing — the only thing under test is the export
itself.

In [3]:
import sys

import cv2

sys.path.append("../src")
import failure_analysis as fa

sample_path = "../assets/_onnx_sanity_frame.jpg"
cap = cv2.VideoCapture("../Anti-UAV-RGBT/test/20190926_134054_1_1/infrared.mp4")
ok, frame = cap.read()
cap.release()
assert ok, "could not read a sample frame from the test video"
cv2.imwrite(sample_path, frame)

pt_result = model.predict(sample_path, verbose=False)[0]
pt_boxes = pt_result.boxes.xyxy.cpu().numpy() if pt_result.boxes is not None else np.empty((0, 4))

onnx_model = YOLO(str(dest))
onnx_result = onnx_model.predict(sample_path, verbose=False)[0]
onnx_boxes = onnx_result.boxes.xyxy.cpu().numpy() if onnx_result.boxes is not None else np.empty((0, 4))

print(f"PyTorch: {len(pt_boxes)} box(es), ONNX: {len(onnx_boxes)} box(es)")


def xyxy_to_xywh(b):
    x1, y1, x2, y2 = b
    return [x1, y1, x2 - x1, y2 - y1]


if len(pt_boxes) and len(onnx_boxes):
    iou = fa.iou_xywh(xyxy_to_xywh(pt_boxes[0]), xyxy_to_xywh(onnx_boxes[0]))
    print(f"IoU between top PyTorch and top ONNX box: {iou:.4f}")
    assert iou > 0.95, "ONNX export diverges from PyTorch output"
    print("Sanity check passed: ONNX output matches PyTorch.")
elif len(pt_boxes) == 0 and len(onnx_boxes) == 0:
    print("Sanity check passed: both backends correctly predict no detection on this frame.")
else:
    raise AssertionError(f"Mismatch: PyTorch found {len(pt_boxes)} boxes, ONNX found {len(onnx_boxes)}")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.


Loading ../web/model/best.onnx for ONNX Runtime inference...


Using ONNX Runtime 1.27.0 with CPUExecutionProvider


PyTorch: 1 box(es), ONNX: 1 box(es)
IoU between top PyTorch and top ONNX box: 0.9881
Sanity check passed: ONNX output matches PyTorch.
